## Flow Cytometry Analysis — Experiment 1
### Author: Eleni Aretaki
### Date: 2026-03-24
### Purpose: Load, preprocess, normalize, and log2-transform flow cytometry data for Experiment 1.

### Experimental design

#### 1. Petri Dishes per Condition:

- Each KO-drug (and WT-drug) combination is grown in its own dish. 
- DMSO is included as a control for both morning and afternoon treatment (same as above, WT-DMSO and KO-DMSO for all cell lines is grown in its own dish).
  
#### 2. 48h Morning vs. Afternoon Treatments:

 Treatments are applied in two separate time windows, which introduces a potential batch effect due to temporal differences (e.g., environmental conditions, handling, etc.). --> We distinguish the treatments based on the **DMSO concentration**.

#### 3. Four Replicates:

The experiment is repeated four times, which provides biological replicates for statistical analysis.

### Downstream Analysis 
#### Analysis of raw data has been performed in **R Studio** using packages for Flow Cytometry analysis, and NOT FlowJo. Relevant scripts can be found on GitHub.

### Normalization

#### In order to count for variability introduced by dishes, time blocks (morning/afternoon) and replicates, we need to normalize our data.
Specifically, we normalize like this:
$$
Normalized Ratio = \frac{Measurement_{ Treated}}{Measurement_{ DMSO}}
$$

This needs to be done for all conditions, WT and KOs for all the different treatments and the WT and DMSO have to derive from the same batch and genotype respectively. This controls for batch-specific variability as well as inherent baseline differences between WT and KO cells.

$$WT_{norm} = \frac{WT_{Treated}}{WT_{DMSO}}$$
$$KO_{norm} = \frac{KO_{Treated}}{KO_{DMSO}}$$
$$Ratio_{KO:WT} = \frac{KO_{norm}}{WT_{norm}}$$

By doing this, we end up with a ratio for WT cells (treated with CPT for example) and a ratio for each KO cell (eg SMARCA4 KO treated with CPT). By taking the **ratio of the ratios**, we end up with our normalized values in respect to their genotype as well as their batch, so we can better see the effects of the drug treatment for one replicate. This ratio tells us the relative effect of the drug in KO cells compared to WT, controlling for variability introduced by the DMSO normalization. Please note that the ratios for WT and KO, are log2-transformed.


## NOTE!
#### For phenotypes like "Apoptosis positive" and "DSB positive", as the untreated samples usually had a low number of cells, calculation of ratios led to misleading results that do NOT represent the flow cytometer scatter plots. Therefore, for these phenotypes, we calculated the *difference* between treated-untreated samples, and not the ratios. Since taking differences, taking the log2 of them would not be meaningful; therefore, we do not need to transform. To compare the KO to WT, we take the **difference of the differences**. 
$$WT_{norm} = {WT_{Treated}}-{WT_{DMSO}}$$
$$KO_{norm} = {KO_{Treated}}-{KO_{DMSO}}$$
$$Ratio_{KO-WT} = {KO_{norm}}-{WT_{norm}}$$
The rest of the analysis remains the same.

##### In total, we have 8 compounds **(Camptothecin, Methyl Methanesulfonate, Adavosertib, Palbociclib, MLN4924, Aphidicolin, Hydroxyurea and BIBR1523)** and 6 main cell lines **(ARID1A, ARID1B, ARID2, SMARCA4, BRD9, and WT)**. 
*PBRM1, SMARCC1, PHF10* and *BRD7* are also screened for some drugs only.

#### It is also important to ensure that the DMSO control values for WT and KOs are consistent across batches and replicates.

#### Another thing to take into account is that apoptotic cells are not only the ones that belong to the "Apoptosis" gate, but also in the "Double Positive" one. Similar goes for double-strand breaks, which are cells belonging to the "DSB positive" gate, but also the "Double Positive" gate. Therefore, when normalizing and calculating ratios, we should adjust the calculations so that the "Apoptosis positive" and "DSB positive" phenotypes include contributions from the "Double positive" phenotype as well. Specifically:

Apoptosis positive should now be calculated as:
$$\text{Apoptosis Positive} = \text{Original Apoptosis Positive} + \text{Double Positive}$$

DSB positive should be calculated as:
$$\text{DSB Positive} = \text{Original DSB Positive} + \text{Double Positive}$$

In [ ]:
# -------------------------------
# Imports
# -------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import matplotlib.cm as cm
from scipy.stats import ttest_ind
from scipy.stats import levene
from statsmodels.stats.multitest import multipletests
import glob
import os

In [ ]:
# -------------------------------
# File paths & constants
# -------------------------------
file_paths = [f"data/raw/Replicate_{i}_countdata.csv" for i in range(1, 5)]
annotation_file = "data/annotation_df.csv"
out_dir = "results/experiment1/"
os.makedirs(out_dir, exist_ok=True)

phenotypes = [
    'S-phase',
    'G1-phase',
    'G2-phase',
    'Negative',
    'Apoptosis-Positive',
    'DSB-Positive',
    'Double-Positive'
]

# Mapping of complicated column names to simpler phenotype names
column_mapping = {
    '/Cells/Singlets/S': 'S-phase',
    '/Cells/Singlets/G1': 'G1-phase',
    '/Cells/Singlets/G2': 'G2-phase',
    '/Cells/Singlets/Negative': 'Negative',
    '/Cells/Singlets/Apoptosis-Positive': 'Apoptosis-Positive',
    '/Cells/Singlets/DSB-Positive': 'DSB-Positive',
    '/Cells/Singlets/Double-Positive': 'Double-Positive'
}

outliers = [(1, 'E8'), (2, 'B4'), (2, 'B3'), (3, 'E12'), (3, 'F12')]
ratio_phenotypes = ['S-phase', 'G1-phase', 'G2-phase']
difference_phenotypes = ['Apoptosis-Positive', 'DSB-Positive']

In [ ]:
# -------------------------------
# Functions
# -------------------------------

def load_annotation(file_path):
    """Load the annotation table."""
    return pd.read_csv(file_path, encoding='unicode_escape')

def load_and_merge_replicates(file_paths, annotation, column_mapping):
    """Load replicate CSVs, pivot percent_of_parent values, merge with annotation, and concatenate."""
    all_data = []
    for i, path in enumerate(file_paths, start=1):
        df = pd.read_csv(path, encoding='unicode_escape')
        percent_pivot = df.pivot(index="well", columns="Population", values="percent_of_parent")
        percent_pivot['replicate'] = i
        merged_df = pd.merge(percent_pivot, annotation, left_on="well", right_on="well_ID", how="inner")
        all_data.append(merged_df)
    combined_df = pd.concat(all_data, ignore_index=True)
    # Keep only needed columns and rename
    columns_to_keep = ["replicate", "well_ID", "modification", "treatment", "DMSO_conc"] + list(column_mapping.keys())
    combined_df = combined_df[columns_to_keep].rename(columns=column_mapping)
    # Adjust Apoptosis-Positive and DSB-Positive
    combined_df['Apoptosis-Positive'] += combined_df['Double-Positive']
    combined_df['DSB-Positive'] += combined_df['Double-Positive']
    return combined_df

def remove_outliers(df, outliers, phenotypes):
    """Flag and remove outlier wells."""
    df = df.copy()
    df['include'] = True
    # Scale values to percentage
    for pheno in phenotypes:
        df[pheno] *= 100
    # Flag outliers
    for replicate, well in outliers:
        df.loc[(df['replicate'] == replicate) & (df['well_ID'] == well), 'include'] = False
    df = df[df['include']].drop(columns='include').reset_index(drop=True)
    return df

def normalize_data(df, ratio_phenotypes, difference_phenotypes):
    """Normalize treated wells using DMSO controls.
    Data are grouped based on the replicate (experiment) they belong to, 
    and the treatment time (morning or afternoon, which is linked to their DMSO concentration 
    - different concentration in the morning that the afternoon)."""
    normalized_data = df.copy()
    rows_to_keep = []

    for (replicate, conc), group in df.groupby(['replicate', 'DMSO_conc']):
        for mod in group['modification'].unique():
            mod_data = group[group['modification'] == mod]
            DMSO_data = mod_data[mod_data['treatment'] == 'DMSO']

            if not DMSO_data.empty:
                dmso_values = DMSO_data.iloc[0]
                treated_mask = (
                    (normalized_data['replicate'] == replicate) &
                    (normalized_data['DMSO_conc'] == conc) &
                    (normalized_data['modification'] == mod) &
                    (normalized_data['treatment'] != 'DMSO')
                )
                for pheno in ratio_phenotypes:
                    normalized_data.loc[treated_mask, pheno] /= dmso_values[pheno]
                for pheno in difference_phenotypes:
                    normalized_data.loc[treated_mask, pheno] -= dmso_values[pheno]
                rows_to_keep.extend(normalized_data[treated_mask].index.tolist())
            else:
                print(f"No DMSO control: Replicate {replicate}, DMSO {conc}, mod {mod}")

    return normalized_data.loc[rows_to_keep].reset_index(drop=True)

def log2_transform(df, ratio_phenotypes, difference_phenotypes):
    """Log2-transform ratio phenotypes, set negative difference phenotypes to 0."""
    transformed_df = df.copy()
    for pheno in ratio_phenotypes:
        transformed_df[pheno] = transformed_df[pheno].apply(lambda x: np.log2(x) if pd.notnull(x) and x > 0 else np.nan)
    for pheno in difference_phenotypes:
        transformed_df[pheno] = transformed_df[pheno].apply(lambda x: 0 if pd.notnull(x) and x < 0 else x)
    return transformed_df

def save_output(df, out_dir, filename):
    """Save dataframe to CSV."""
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, filename), index=False)

In [ ]:
# -------------------------------
# Main pipeline
# -------------------------------
if __name__ == "__main__":
    annotation = load_annotation(annotation_file)
    combined_df = load_and_merge_replicates(file_paths, annotation, column_mapping)
    cleaned_df = remove_outliers(combined_df, outliers, phenotypes)
    normalized_df = normalize_data(cleaned_df, ratio_phenotypes, difference_phenotypes)
    log2_df = log2_transform(normalized_df, ratio_phenotypes, difference_phenotypes)

    # Save outputs
    save_output(cleaned_df, out_dir, "fc1_combined_data.csv")
    save_output(log2_df, out_dir, "fc1_normalized_data.csv")